<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the Python libraries the notebook needs. Run it once, then restart the kernel before you import anything.

In [ ]:
!pip install pandas scipy matplotlib alphalens-reloaded

Zipline Reloaded is left out on purpose because it pins specific NumPy and pandas versions and needs a price bundle downloaded before it will run. Install it on its own with conda install -c conda-forge zipline-reloaded, or into a clean Python 3.9 or 3.10 environment, otherwise the resolver will try to downgrade the pandas and NumPy already installed to satisfy those pins.

## Imports and setup

We use warnings to suppress console output, pandas to reshape the recorded output into tables, scipy for its statistics helpers, and matplotlib to draw the charts. Zipline runs the backtest and computes the weekly ranking, and AlphaLens measures how closely that ranking lined up with the returns that came after it.

In [ ]:
import warnings

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from alphalens.performance import factor_information_coefficient
from alphalens.plotting import plot_ic_ts
from alphalens.utils import (get_clean_factor_and_forward_returns,
                             get_forward_returns_columns)
from scipy import stats
from zipline import run_algorithm
from zipline.api import (attach_pipeline, calendars, date_rules,
                         get_open_orders, order_target_percent,
                         pipeline_output, record, schedule_function,
                         set_commission, set_slippage, time_rules)
from zipline.pipeline import CustomFactor, Pipeline
from zipline.pipeline.data import USEquityPricing

A few of these are never called, including stats, get_forward_returns_columns, set_commission and set_slippage. The bound names sit idle, though importing scipy.stats still costs load time and memory, so knowing which ones the code touches makes the file easier to strip down later. factor_information_coefficient computes the score and plot_ic_ts draws it.

Silence every warning Python would print, not just the deprecation notices, and fix how many names sit on each side of the book.

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
N_LONGS = N_SHORTS = 10

Ten names per side puts 10% of capital in each position, and the statistic we read at the end is the rank correlation, not the equity curve. We aren't tuning that number today, because we're scoring the ranking itself rather than the portfolio built on top of it.

Load the Zipline notebook extension and download the Quandl bundle of daily US equity prices to local storage.

In [ ]:
%load_ext zipline
!zipline ingest -b quandl

The ingest step downloads the WIKI daily bars and took a few minutes on my laptop the first time, though that depends on your connection. The free Quandl bundle stopped updating in March 2018, which is why the backtest below ends before then. Everything after this point reads from that local copy, so you can rerun the notebook offline.

## Define the momentum ranking factor

Define the score we're going to test. It divides the latest close by the close at the start of the window, giving the price ratio over that many trading days.

In [ ]:
class Momentum(CustomFactor):

    inputs = [USEquityPricing.close]

    def compute(self, today, assets, out, close):
        out[:] = close[-1] / close[0]

A value above 1 means the stock rose over the window. Keeping the formula this plain matters, because a correlation near zero then points to weak prediction rather than an arithmetic mistake in the factor.

Build the weekly ranking. The pipeline computes 20-day and 30-day price ratios, keeps only stocks positive on both, and ranks the survivors by their 30-day figure.

In [ ]:
def make_pipeline():
    twenty_day_momentum = Momentum(window_length=20)
    thirty_day_momentum = Momentum(window_length=30)

    positive_momentum = (twenty_day_momentum > 1) & (thirty_day_momentum > 1)

    return Pipeline(
        columns={
            "longs": thirty_day_momentum.top(N_LONGS),
            "shorts": thirty_day_momentum.top(N_SHORTS),
            "ranking": thirty_day_momentum.rank(ascending=False),
        },
        screen=positive_momentum,
    )

Notice that the shorts column calls .top() rather than .bottom(), so it picks the same ten names as the longs column, and because the long order is already pending when exec_trades reaches the shorts, get_open_orders blocks the short and the book stays long. The score we compute at the end reads the ranking column and not the trades, so the measurement holds either way. The screen also removes every stock with negative momentum, so we only ever rank names that already went up.

## Run the backtest with Zipline

Pull the pipeline output into the algorithm each morning and schedule the rebalance for the market open on the first trading day of each week.

In [ ]:
def before_trading_start(context, data):
    context.factor_data = pipeline_output("factor_pipeline")

In [ ]:
def initialize(context):
    attach_pipeline(make_pipeline(), "factor_pipeline")
    schedule_function(
        rebalance,
        date_rules.week_start(),
        time_rules.market_open(),
        calendar=calendars.US_EQUITIES,
    )

The rebalance runs weekly and we only record prices on rebalance dates, so AlphaLens receives a weekly price table and a period of 5 counts five rows, not five trading days. To read genuine 5-day returns you would have to hand it a daily price history pulled from the bundle instead.

Rebalance the book, and record the ranking and the prices on every rebalance date so we can score them afterward.

In [ ]:
def rebalance(context, data):
    factor_data = context.factor_data
    record(factor_data=factor_data.ranking)

    assets = factor_data.index
    record(prices=data.current(assets, "price"))

    longs = assets[factor_data.longs]
    shorts = assets[factor_data.shorts]
    divest = set(context.portfolio.positions.keys()) - set(
        longs.union(shorts)
    )

    exec_trades(data, assets=divest, target_percent=0)
    exec_trades(data, assets=longs, target_percent=1 / N_LONGS)
    exec_trades(data, assets=shorts, target_percent=-1 / N_SHORTS)

In [ ]:
def exec_trades(data, assets, target_percent):
    for asset in assets:
        if data.can_trade(asset) and not get_open_orders(asset):
            order_target_percent(asset, target_percent)

The record() calls persist the ranks and the prices, which are the only two inputs AlphaLens needs later. The Zipline examples I've worked through stop at the equity curve that run_algorithm returns, and portfolio returns mix the quality of the ranking with position sizing and execution. Saving the raw ranks each week separates the two.

Run the backtest from January 1, 2015 to January 1, 2018, since pd.Timestamp("2018") resolves to the first day of that year, with $100,000 of starting capital.

In [ ]:
start = pd.Timestamp("2015")
end = pd.Timestamp("2018")
perf = run_algorithm(
    start=start,
    end=end,
    initialize=initialize,
    before_trading_start=before_trading_start,
    capital_base=100_000,
    bundle="quandl",
)

Three years of weekly rebalances gives us roughly 150 ranking dates, which is thin for judging a score that sits near zero. The bundle holds plenty of earlier history, so an earlier start date is available and I keep the window short only to make reruns fast, which makes this a working example of the measurement rather than a verdict on momentum.

## Score the ranking with AlphaLens

Turn the recorded prices into a table with dates down the rows and ticker symbols across the columns.

In [ ]:
prices = pd.concat(
    [df.to_frame(d) for d, df in perf.prices.dropna().items()],
    axis=1,
).T
prices.columns = [col.symbol for col in prices.columns]
prices.index = prices.index.normalize()

AlphaLens expects prices in exactly this shape, and normalize() strips the time off each timestamp so the price dates match the ranking dates. When they don't match, get_clean_factor_and_forward_returns drops the unmatched rows and prints how many it dropped, so read that count before you trust the chart.

Reshape the recorded rankings the same way, then stack them into one long column indexed by date and asset.

In [ ]:
factor_data = pd.concat(
    [df.to_frame(d) for d, df in perf.factor_data.dropna().items()],
    axis=1,
).T
factor_data.columns = [col.symbol for col in factor_data.columns]
factor_data.index = factor_data.index.normalize()
factor_data = factor_data.stack()
factor_data.index.names = ["date", "asset"]

The scores go in as a single series with a two-level index named date and asset, and handing AlphaLens the wide frame instead raises an index error. Note also that rank(ascending=False) assigns the strongest momentum name the value 1, so small factor values mark the best stocks and a negative correlation is the encouraging reading here. The stack() call also drops the blanks for stocks the screen excluded that week, so only ranked names carry through.

Join each week's ranks to the returns that followed over the next 5, 10, 21 and 63 rows of the weekly price table, and sort the names into five buckets by rank.

In [ ]:
alphalens_data = get_clean_factor_and_forward_returns(
    factor=factor_data,
    prices=prices,
    periods=(5, 10, 21, 63),
    quantiles=5,
)

This one function handles the date alignment, which is fiddly to write by hand and easy to get wrong by a single row. It pairs each rank with returns that happen after the ranking date, never before, which is the difference between a real test and one that uses prices you couldn't have seen. The four windows show how the correlation changes as the holding period lengthens.

Compute the information coefficient, which is the rank correlation between the ranks assigned on a date and the returns that followed, running from -1 to 1.

In [ ]:
ic = factor_information_coefficient(alphalens_data)

The result has one row per rebalance date and one column per forward window, so we can read the score through time instead of collapsing it to one average. I won't quote a benchmark number here, because the values people cite depend on the universe, the horizon and the weighting scheme, so compare the four columns against each other instead.

Print ic.columns first to confirm the forward-return column names, then plot the shortest window date by date with a rolling average over it.

In [ ]:
plot_ic_ts(ic[["5D"]])
plt.tight_layout()

An average across the whole sample hides the stretches where the correlation turned negative, and the date-by-date plot shows those stretches directly. Momentum funds are widely reported to have reversed hard in the spring of 2009, and our sample starts in 2015, so no reading here covers that.

Average the score within each calendar year and plot all four forward windows side by side.

In [ ]:
ic_by_year = ic.resample("A").mean()
ic_by_year.index = ic_by_year.index.year
ic_by_year.plot.bar(figsize=(14, 6))
plt.tight_layout()
plt.show()

The yearly bars set up two comparisons. One is 2015 against 2016 and 2017, years with quite different index volatility, and the other is the shortest window against the 21 and 63-period bars inside each year. If the short-horizon correlation comes out larger than the long-horizon one, that pattern points toward frequent rebalances, though turnover and commissions decide whether any of it survives as profit.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.